In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
import numpy as np
import evaluate
import matplotlib.pyplot as plt

In [ ]:
dataset = load_dataset("tweet_eval", "sentiment")
print(dataset)

In [ ]:
model_id = "model/base-60000"
sentiment_model_id = "classifier"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=3,
    ignore_mismatched_sizes=True,
)

model_size = sum(t.numel() for t in model.parameters())
print(f"Transformer size: {model_size/1000**2:.1f}M parameters")

label_names = dataset["train"].features["label"].names
model.config.id2label = {i: name for i, name in enumerate(label_names)}
model.config.label2id = {name: i for i, name in enumerate(label_names)}

In [ ]:
def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=64,
    )

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"])
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch")
dataset

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1_macro = f1.compute(
        predictions=preds,
        references=labels,
        average="macro"
    )["f1"]

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
    }

training_args = TrainingArguments(
    output_dir=sentiment_model_id,
    learning_rate=2e-5,
    weight_decay=0.02,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    logging_steps=500,
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=True,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(sentiment_model_id)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)

In [ ]:
logs = trainer.state.log_history

train_steps = [log["step"] for log in logs if "loss" in log and "eval_loss" not in log]
train_losses = [log["loss"] for log in logs if "loss" in log and "eval_loss" not in log]

eval_steps = [log["step"] for log in logs if "eval_loss" in log]
eval_losses = [log["eval_loss"] for log in logs if "eval_loss" in log]

plt.figure(figsize=(12, 6))
plt.plot(train_steps, train_losses, label="Training Loss", linewidth=2)
plt.plot(eval_steps, eval_losses, label="Validation Loss", linewidth=2)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

In [ ]:
test_metrics = trainer.evaluate(dataset["test"])
print(test_metrics)

In [ ]:
from transformers import pipeline

classifier = pipeline(
    task="text-classification",
    model=sentiment_model_id,
    tokenizer=sentiment_model_id,
)

print(classifier("I love this phone"))
print(classifier("This is the worst service ever"))